# GeoDiff-GAN DGX 25MCSA19: Recent SOTA Benchmarks

Trains official x4 SISR/remote-sensing SR repositories on the same
Sentinel-2 128x128 -> 512x512 task. Competitor source code, published
upsampling heads, input size and output size are not changed in
`ARCHITECTURE_MODE='official'`.

## 0. Create the local Python 3.11 environment

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

ASSIGNED_ROOT = Path("/workspace/temp/25mcsa19/working/temp/25mcsa19").resolve()
ASSIGNED_ROOT.mkdir(parents=True, exist_ok=True)
VENV = ASSIGNED_ROOT / ".venvs" / "geodiff-py311"
KERNEL_DIR = ASSIGNED_ROOT / "jupyter_kernels" / "geodiff-py311-25mcsa19"

def run(command, check=True, cwd=None):
    command = [str(value) for value in command]
    print("+", " ".join(command), flush=True)
    return subprocess.run(command, check=check, cwd=cwd)

uv = shutil.which("uv")
if uv is None:
    BOOTSTRAP = ASSIGNED_ROOT / ".bootstrap"
    run([sys.executable, "-m", "pip", "install", "--prefix", BOOTSTRAP, "uv"])
    uv = str(BOOTSTRAP / "bin" / "uv")
if not Path(uv).exists() and shutil.which(uv) is None:
    raise RuntimeError(f"uv was not found at {uv}. Ask the administrator to provide uv or Python 3.11.")

run([uv, "python", "install", "3.11"])
run([uv, "venv", "--python", "3.11", "--seed", VENV])
PYTHON311 = VENV / "bin" / "python"
run([PYTHON311, "-m", "ensurepip", "--upgrade"], check=False)
run([PYTHON311, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel", "ipykernel"])

KERNEL_DIR.mkdir(parents=True, exist_ok=True)
(KERNEL_DIR / "kernel.json").write_text(json.dumps({
    "argv": [str(PYTHON311), "-m", "ipykernel_launcher", "-f", "{connection_file}"],
    "display_name": "GeoDiff-GAN 25MCSA19 Python 3.11",
    "language": "python",
    "env": {
        "PYTHONUNBUFFERED": "1",
        "XDG_CACHE_HOME": str(ASSIGNED_ROOT / ".cache"),
        "HF_HOME": str(ASSIGNED_ROOT / ".cache" / "huggingface"),
        "TORCH_HOME": str(ASSIGNED_ROOT / ".cache" / "torch"),
        "KAGGLE_CONFIG_DIR": str(ASSIGNED_ROOT / "secrets" / "kaggle"),
        "KAGGLEHUB_CACHE": str(ASSIGNED_ROOT / "downloads" / "kagglehub_cache")
    }
}, indent=2), encoding="utf-8")

print("Created Python:", PYTHON311)
print("Created local kernelspec:", KERNEL_DIR)
print("No files outside the assigned root were modified by default.")
print()
print("If this kernel is not visible in Jupyter, run this optional command manually:")
print(f"{PYTHON311} -m ipykernel install --user --name geodiff-py311-25mcsa19 --display-name 'GeoDiff-GAN 25MCSA19 Python 3.11'")
print("Then switch Kernel ->", "GeoDiff-GAN 25MCSA19 Python 3.11", "and rerun from the runtime cell.")

## 1. Runtime root guard and repository install

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

ASSIGNED_ROOT = Path("/workspace/temp/25mcsa19/working/temp/25mcsa19").resolve()
if not str(ASSIGNED_ROOT).startswith("/workspace/temp/25mcsa19/working/temp/25mcsa19"):
    raise RuntimeError(f"Unsafe root: {ASSIGNED_ROOT}")
ASSIGNED_ROOT.mkdir(parents=True, exist_ok=True)

THESIS_ROOT = ASSIGNED_ROOT
REPOSITORY_DIR = THESIS_ROOT / "geodiff-gan"
DOWNLOAD_ROOT = THESIS_ROOT / "downloads"
DATASET_ROOT = THESIS_ROOT / "datasets" / "sentinel2-bharat"
WORK_ROOT = THESIS_ROOT / "geodiff-output"
SOURCE_ROOT = THESIS_ROOT / "sota_sources"
BACKUP_ROOT = THESIS_ROOT / "backups"
REPOSITORY_URL = "https://github.com/shashankjs2002/SI-SR-1.git"
KAGGLE_DATASET = "twilight2002/sentinel2-bharat"

for path in (DOWNLOAD_ROOT, DATASET_ROOT, WORK_ROOT, SOURCE_ROOT, BACKUP_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["XDG_CACHE_HOME"] = str(THESIS_ROOT / ".cache")
os.environ["HF_HOME"] = str(THESIS_ROOT / ".cache" / "huggingface")
os.environ["TRANSFORMERS_CACHE"] = str(THESIS_ROOT / ".cache" / "huggingface")
os.environ["TORCH_HOME"] = str(THESIS_ROOT / ".cache" / "torch")
os.environ["KAGGLE_CONFIG_DIR"] = str(THESIS_ROOT / "secrets" / "kaggle")
os.environ["KAGGLEHUB_CACHE"] = str(DOWNLOAD_ROOT / "kagglehub_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def assert_inside(path):
    path = Path(path).resolve()
    if not str(path).startswith(str(THESIS_ROOT)):
        raise RuntimeError(f"Refusing to touch path outside assigned root: {path}")
    return path

def run(command, cwd=None, env=None, check=True):
    command = [str(value) for value in command]
    environment = os.environ.copy()
    if env:
        environment.update(env)
    print("+", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, env=environment, check=check)

PYTHON = Path(sys.executable)
PIP = [PYTHON, "-m", "pip"]
print("Python:", sys.version)
print("Executable:", PYTHON)
print("Assigned root:", THESIS_ROOT)
if sys.version_info < (3, 10):
    raise RuntimeError("Switch to the GeoDiff-GAN 25MCSA19 Python 3.11 kernel before continuing.")

In [ ]:
import os, sys
from pathlib import Path

if REPOSITORY_DIR.exists():
    if (REPOSITORY_DIR / ".git").exists():
        print("Updating existing Git clone:", REPOSITORY_DIR)
        run(["git", "pull", "--ff-only"], cwd=REPOSITORY_DIR, check=False)
    elif (REPOSITORY_DIR / "pyproject.toml").exists() and (REPOSITORY_DIR / "src").exists():
        print("Using existing uploaded source tree:", REPOSITORY_DIR)
    else:
        raise RuntimeError(f"{REPOSITORY_DIR} exists but is not a GeoDiff-GAN source tree.")
else:
    run(["git", "clone", "--depth", "1", REPOSITORY_URL, REPOSITORY_DIR])

run([*PIP, "install", "--upgrade", "pip", "setuptools", "wheel"])
run([*PIP, "install", "numpy>=1.26", "Pillow>=10", "PyYAML>=6", "tqdm>=4.66", "rasterio>=1.3", "pandas>=2", "matplotlib>=3.8", "kaggle", "kagglehub", "ipykernel"])
run([*PIP, "install", "-e", ".", "--no-deps"], cwd=REPOSITORY_DIR)
sys.path.insert(0, str(REPOSITORY_DIR / "src"))
os.chdir(REPOSITORY_DIR)
run(["git", "log", "-1", "--oneline"], cwd=REPOSITORY_DIR, check=False)

In [ ]:
import torch

print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available(), "visible GPUs:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Ask the administrator to check the DGX Jupyter container.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("Exactly one GPU must be visible. Coordinate with other users and set CUDA_VISIBLE_DEVICES before starting.")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, "VRAM GiB:", round(props.total_memory / 1024**3, 2))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

## 2. Clone official SOTA repositories inside the assigned folder

In [ ]:
from geodiff_gan.benchmark.models import MODEL_SPECS

run([*PIP, "install", "timm>=1.0.15", "einops>=0.8", "pandas>=2", "matplotlib>=3.8"])

SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
REPOSITORIES = {key: spec.repository for key, spec in MODEL_SPECS.items()}
DIRECTORIES = {key: spec.directory for key, spec in MODEL_SPECS.items()}
for key, url in REPOSITORIES.items():
    destination = SOURCE_ROOT / DIRECTORIES[key]
    if destination.exists():
        print("Keeping existing source:", key, destination)
    else:
        run(["git", "clone", "--depth", "1", url, destination])

## 3. Load the shared Sentinel-2 split

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F
from geodiff_gan.data import SentinelPatchDataset

candidate_manifests = [
    WORK_ROOT / "manifest_ms_80_10_10.jsonl",
    WORK_ROOT / "manifest_dgx_80_10_10.jsonl",
    WORK_ROOT / "manifest_ms_raw.jsonl",
    WORK_ROOT / "manifest_raw.jsonl",
    WORK_ROOT / "manifest.jsonl",
]
MANIFEST = next((path for path in candidate_manifests if path.exists()), None)
if MANIFEST is None:
    raise FileNotFoundError("Run the multispectral variant notebook preprocessing first.")
records = [json.loads(line) for line in MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
split_counts = Counter(record["split"] for record in records)
print("Manifest:", MANIFEST)
print("Split counts:", split_counts)
if any(split_counts[name] == 0 for name in ("train", "val", "test")):
    raise RuntimeError(f"Manifest must contain train/val/test records: {split_counts}")

dataset = SentinelPatchDataset(MANIFEST, split="train", scale=4, caption_file=None, augment=False, random_degradation=False, degradation_seed=42, degradation_severity="mild")
sample = dataset[0]
lr = sample["lr_rgb"] if "lr_rgb" in sample else sample["lr"]
hr = sample["hr"]
bicubic = F.interpolate(lr[None], size=hr.shape[-2:], mode="bicubic", align_corners=False)[0].clamp(0, 1)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, image, title in zip(axes, [lr, bicubic, hr], ["LR 128x128", "Bicubic 512x512", "Target 512x512"]):
    axis.imshow(image.clamp(0, 1).permute(1, 2, 0))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 4. Select official benchmark profile

In [ ]:
PROFILE = "screening"  # smoke | screening | paper
ARCHITECTURE_MODE = "official"
LR_CROP = 128          # keeps the task as 128x128 -> 512x512
INCLUDE_MAMBA_MODELS = False
MODELS_TO_RUN = [
    "swinir", "hat", "srformer", "dat", "omnisr",
    "ttst", "mfghmoe", "swin2mose", "atd",
]
if INCLUDE_MAMBA_MODELS:
    MODELS_TO_RUN += ["mambair", "mambairv2", "fremamba"]

PROFILES = {
    "smoke": {"max_updates": 20, "validate_every": 10, "validation_limit": 4, "test_limit": 4, "early_stopping_patience": 2},
    "screening": {"max_updates": 5000, "validate_every": 500, "validation_limit": 32, "test_limit": 40, "early_stopping_patience": 4},
    "paper": {"max_updates": 50000, "validate_every": 2000, "validation_limit": 128, "test_limit": 80, "early_stopping_patience": 6},
}
SETTINGS = PROFILES[PROFILE]
BATCH_SETTINGS = {name: (1, 8) for name in MODELS_TO_RUN}
BATCH_SETTINGS["omnisr"] = (2, 4)
BENCHMARK_ROOT = WORK_ROOT / "sota_benchmark"
print("Models:", MODELS_TO_RUN)
print("Official architecture mode:", ARCHITECTURE_MODE)
print("LR crop:", LR_CROP, "=> output", LR_CROP * 4)

## 5. Optional Mamba dependencies

In [ ]:
if INCLUDE_MAMBA_MODELS:
    result = run([PYTHON, "-m", "pip", "install", "causal-conv1d>=1.4.0", "mamba-ssm>=2.2.0", "thop"], check=False)
    if result.returncode != 0:
        raise RuntimeError("Mamba dependencies failed. Set INCLUDE_MAMBA_MODELS=False.")
else:
    print("Mamba models disabled; no CUDA extension installation needed.")

## 6. Probe input/output and architecture before training

In [ ]:
PROBE_RESULTS = {}
for model_name in MODELS_TO_RUN:
    command = [PYTHON, "-m", "geodiff_gan.cli.benchmark_sota", "--model", model_name, "--source-root", SOURCE_ROOT, "--manifest", MANIFEST, "--output", BENCHMARK_ROOT / ARCHITECTURE_MODE / model_name, "--architecture-mode", ARCHITECTURE_MODE, "--probe-only"]
    completed = subprocess.run([str(value) for value in command], cwd=REPOSITORY_DIR, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Probe failed for {model_name}")
    PROBE_RESULTS[model_name] = json.loads(completed.stdout[completed.stdout.index("{"):])
print("All probes passed.")

## 7. Train models sequentially with automatic resume

In [ ]:
TRAINING_RESULTS = {}
for model_name in MODELS_TO_RUN:
    batch_size, accumulation = BATCH_SETTINGS[model_name]
    output = BENCHMARK_ROOT / ARCHITECTURE_MODE / PROFILE / model_name
    command = [
        PYTHON, "-m", "geodiff_gan.cli.benchmark_sota",
        "--model", model_name,
        "--source-root", SOURCE_ROOT,
        "--manifest", MANIFEST,
        "--output", output,
        "--architecture-mode", ARCHITECTURE_MODE,
        "--max-updates", SETTINGS["max_updates"],
        "--batch-size", batch_size,
        "--accumulation", accumulation,
        "--learning-rate", 2e-4,
        "--weight-decay", 1e-4,
        "--lr-crop", LR_CROP,
        "--num-workers", 6,
        "--validation-limit", SETTINGS["validation_limit"],
        "--test-limit", SETTINGS["test_limit"],
        "--validate-every", SETTINGS["validate_every"],
        "--early-stopping-patience", SETTINGS["early_stopping_patience"],
        "--degradation-seed", 42,
        "--degradation-severity", "mild",
        "--seed", 42,
    ]
    print("\n" + "=" * 80)
    print("Training", model_name, "->", output)
    print("=" * 80)
    run(command, cwd=REPOSITORY_DIR)
    TRAINING_RESULTS[model_name] = json.loads((output / "test_metrics.json").read_text(encoding="utf-8"))

## 8. Compare SOTA and GeoDiff-GAN metrics

In [ ]:
import pandas as pd

rows = []
for model_name in MODELS_TO_RUN:
    output = BENCHMARK_ROOT / ARCHITECTURE_MODE / PROFILE / model_name
    metrics = json.loads((output / "test_metrics.json").read_text(encoding="utf-8"))
    run_info = json.loads((output / "run.json").read_text(encoding="utf-8"))
    rows.append({"family": "SOTA official", "method": model_name, "architecture_mode": run_info["architecture_mode"], "parameters": run_info["parameters"], **metrics})

for variant in ("small_improved", "medium", "small_improved_ms", "medium_ms"):
    metrics_path = WORK_ROOT / "evaluation" / variant / "test" / "metrics.json"
    if metrics_path.exists():
        rows.append({"family": "GeoDiff-GAN", "method": f"geodiff_{variant}", **json.loads(metrics_path.read_text(encoding="utf-8"))})

comparison = pd.DataFrame(rows)
preferred = ["family", "method", "architecture_mode", "parameters", "count", "l1", "psnr", "ssim", "edge_f1", "redegradation_l1", "lpips", "dists"]
comparison = comparison[[name for name in preferred if name in comparison.columns]].sort_values("psnr", ascending=False)
display(comparison.round(6))
out = BENCHMARK_ROOT / ARCHITECTURE_MODE / PROFILE / "all_model_comparison.csv"
out.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(out, index=False)
print("Wrote:", out)

## 9. Side-by-side qualitative outputs

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

image_roots = {name: BENCHMARK_ROOT / ARCHITECTURE_MODE / PROFILE / name / "images" / "test" for name in MODELS_TO_RUN}
available = [name for name, root in image_roots.items() if list(root.glob("*_hr.png"))]
if not available:
    print("No saved qualitative outputs yet.")
else:
    reference_model = available[0]
    reference_files = sorted(image_roots[reference_model].glob("*_hr.png"))
    for reference_hr in reference_files[:5]:
        prefix = reference_hr.name.removesuffix("_hr.png")
        lr_path = image_roots[reference_model] / f"{prefix}_lr.png"
        panels = [("Observed LR", Image.open(lr_path)), ("Target HR", Image.open(reference_hr))]
        for model_name, root in image_roots.items():
            path = root / f"{prefix}_sr.png"
            if path.exists():
                panels.append((model_name, Image.open(path)))
        columns = 4
        rows_count = int(np.ceil(len(panels) / columns))
        figure, axes = plt.subplots(rows_count, columns, figsize=(16, 4 * rows_count))
        axes = np.asarray(axes).reshape(-1)
        for axis, (title, image) in zip(axes, panels):
            axis.imshow(image)
            axis.set_title(title)
            axis.axis("off")
        for axis in axes[len(panels):]:
            axis.axis("off")
        figure.suptitle(prefix)
        plt.tight_layout()
        plt.show()

## 10. Backup benchmark outputs

In [ ]:
archive_base = BACKUP_ROOT / f"geodiff_multispectral_variants_{time.strftime('%Y%m%d_%H%M%S')}"
archive_path = shutil.make_archive(str(archive_base), "gztar", root_dir=WORK_ROOT)
print("Backup archive:", archive_path)
print("No dataset or checkpoint files were deleted.")